# Lab type: debug# Course: ML203 — Unsupervised Learning & Clustering# Lesson: DBSCAN# Task: Find and fix the 3 bugs in the DBSCAN pipeline. After fixing each bug, write a one-sentence explanation.

In [ ]:
import numpy as npimport pandas as pdfrom sklearn.preprocessing import StandardScalerfrom sklearn.cluster import DBSCANfrom sklearn.neighbors import NearestNeighborsimport matplotlib.pyplot as plt

## Step 1: Generate Non-Spherical Data

In [ ]:
np.random.seed(42)# Create two crescent-shaped clusters (k-means would fail here)theta = np.linspace(0, 2*np.pi, 50)cluster_1 = np.column_stack([5*np.cos(theta) + np.random.normal(0, 0.3, 50),                              5*np.sin(theta) + np.random.normal(0, 0.3, 50)])cluster_2 = np.column_stack([10*np.cos(theta) + np.random.normal(0, 0.3, 50),                              10*np.sin(theta) + np.random.normal(0, 0.3, 50)])X = np.vstack([cluster_1, cluster_2])scaler = StandardScaler()X_scaled = scaler.fit_transform(X)print(f"Data shape: {X_scaled.shape}")

## Step 2: Bug 1 — Incorrect eps Parameter

In [ ]:
# BUG 1: eps is too large (or too small), not calibrated to data scaledbscan = DBSCAN(eps=100, min_samples=5)  # BUG: eps=100 is way too largelabels = dbscan.fit_predict(X_scaled)print(f"Number of clusters: {len(set(labels)) - (1 if -1 in labels else 0)}")print(f"Number of noise points: {sum(labels == -1)}")

**Fix explanation here:**

## Step 3: Bug 2 — Missing eps Calibration

In [ ]:
# BUG 2: eps is chosen without checking the data# (The k-distance graph is the standard way to calibrate eps)dbscan = DBSCAN(eps=0.5, min_samples=5)  # BUG: Chosen arbitrarily without validationlabels = dbscan.fit_predict(X_scaled)print(f"Number of clusters: {len(set(labels)) - (1 if -1 in labels else 0)}")print(f"Number of noise points: {sum(labels == -1)}")

**Fix explanation here:**

## Step 4: Bug 3 — Ignoring Noise Points

In [ ]:
dbscan = DBSCAN(eps=0.3, min_samples=5)labels = dbscan.fit_predict(X_scaled)# BUG 3: Analysis ignores noise pointsunique_labels = set(labels)print(f"Unique labels (including noise): {unique_labels}")for label in sorted(unique_labels):    cluster_points = sum(labels == label)    # BUG: This treats noise (-1) same as real clusters    print(f"Cluster {label}: {cluster_points} points")

**Fix explanation here:**

## Step 5: Corrected Pipeline with eps Calibration

In [ ]:
# The correct approach calibrates eps using the k-distance graph:k = 5  # min_samples parameterneighbors = NearestNeighbors(n_neighbors=k)neighbors_fit = neighbors.fit(X_scaled)distances, indices = neighbors_fit.kneighbors(X_scaled)# Sort distances to the k-th nearest neighbordistances = np.sort(distances[:, k-1], axis=0)# Plot the k-distance graphplt.figure(figsize=(8, 5))plt.plot(distances)plt.xlabel('Data Points sorted by distance')plt.ylabel(f'{k}-th Nearest Neighbor Distance')plt.title('K-distance Graph (helps choose eps)')plt.grid(True, alpha=0.3)plt.show()print(f"Looking for the elbow in the plot above to choose eps")

## SummaryDBSCAN is powerful for non-spherical clusters, but requires careful calibration:- **eps** must be chosen using the k-distance graph, not guessed- **min_samples** should relate to your data density and desired cluster size- **Noise points** (-1) are legitimate and must be handled**Next lesson:** Hierarchical Clustering — explore cluster structure at multiple levels.